In [1]:
import os
import pandas as pd
import numpy as np

RAW_DIR = "../data/raw"
PROC_DIR = "../data/processed"

future = pd.read_csv(os.path.join(RAW_DIR, "future_picks_2025_26.csv"))
team_roster = pd.read_csv(os.path.join(PROC_DIR, "team_total_value_2025_26.csv"))

print("future shape:", future.shape)
print("team_roster shape:", team_roster.shape)

future.head()

future shape: (420, 5)
team_roster shape: (30, 2)


,team,pick_year,round,expected_pick_number,protection_note
0,Atlanta Hawks,2026,1,NaN,More favorable of (i) less favorable of ATL an...
1,Atlanta Hawks,2026,2,NaN,More favorable of (i) ATL and (ii) less favora...
2,Atlanta Hawks,2027,1,NaN,To SAN; Less favorable of MIL and NOP if eithe...
3,Atlanta Hawks,2027,2,NaN,Own (via BOS to MEM to POR)
4,Atlanta Hawks,2028,1,NaN,More favorable of (i) ATL and (ii) less favora...


In [2]:
future["team"] = future["team"].astype(str).str.strip()
team_roster["team"] = team_roster["team"].astype(str).str.strip()

future["round"] = pd.to_numeric(future["round"], errors="coerce")
future["pick_year"] = pd.to_numeric(future["pick_year"], errors="coerce")

future[["team", "pick_year", "round"]].head()

,team,pick_year,round
0,Atlanta Hawks,2026,1
1,Atlanta Hawks,2026,2
2,Atlanta Hawks,2027,1
3,Atlanta Hawks,2027,2
4,Atlanta Hawks,2028,1


In [3]:
team_strength = team_roster.sort_values("total_value", ascending=True).reset_index(drop=True)
team_strength["strength_rank"] = team_strength.index + 1

team_strength[["team", "total_value", "strength_rank"]].head(10)

,team,total_value,strength_rank
0,Washington Wizards,3526.320257,1
1,Sacramento Kings,8531.700275,2
2,Utah Jazz,10105.950256,3
3,Brooklyn Nets,12296.040293,4
4,Indiana Pacers,13790.410282,5
5,Dallas Mavericks,16183.250427,6
6,New Orleans Pelicans,16858.440276,7
7,Memphis Grizzlies,17336.950149,8
8,Chicago Bulls,17638.400200,9
9,Milwaukee Bucks,18692.900537,10


In [4]:
rank_map = dict(zip(team_strength["team"], team_strength["strength_rank"]))

future["strength_rank"] = future["team"].map(rank_map)

print("Missing strength ranks:", future["strength_rank"].isna().sum())

future[["team", "pick_year", "round", "strength_rank"]].head(10)

Missing strength ranks: 0


,team,pick_year,round,strength_rank
0,Atlanta Hawks,2026,1,23
1,Atlanta Hawks,2026,2,23
2,Atlanta Hawks,2027,1,23
3,Atlanta Hawks,2027,2,23
4,Atlanta Hawks,2028,1,23
5,Atlanta Hawks,2028,2,23
6,Atlanta Hawks,2029,1,23
7,Atlanta Hawks,2029,2,23
8,Atlanta Hawks,2030,1,23
9,Atlanta Hawks,2030,2,23


In [5]:
def assign_pick_number(rank, round_):
    rank = int(rank)
    round_ = int(round_)
    
    if round_ == 1:
        return rank          # 1 to 30
    elif round_ == 2:
        return rank + 30     # 31 to 60
    else:
        return np.nan

future["expected_pick_number"] = future.apply(
    lambda row: assign_pick_number(row["strength_rank"], row["round"]),
    axis=1
)

future[["team", "strength_rank", "round", "expected_pick_number"]].head(15)

,team,strength_rank,round,expected_pick_number
0,Atlanta Hawks,23,1,23
1,Atlanta Hawks,23,2,53
2,Atlanta Hawks,23,1,23
3,Atlanta Hawks,23,2,53
4,Atlanta Hawks,23,1,23
5,Atlanta Hawks,23,2,53
6,Atlanta Hawks,23,1,23
7,Atlanta Hawks,23,2,53
8,Atlanta Hawks,23,1,23
9,Atlanta Hawks,23,2,53


In [6]:
future.groupby("team")["expected_pick_number"].mean().sort_values().head(10)

team
Washington Wizards      16.0
Sacramento Kings        17.0
Utah Jazz               18.0
Brooklyn Nets           19.0
Indiana Pacers          20.0
Dallas Mavericks        21.0
New Orleans Pelicans    22.0
Memphis Grizzlies       23.0
Chicago Bulls           24.0
Milwaukee Bucks         25.0
Name: expected_pick_number, dtype: float64

In [7]:
def ev_points(pick):
    return 110 / (1 + (pick / 6))

future["ev_points"] = future["expected_pick_number"].apply(ev_points)

future[["expected_pick_number", "ev_points"]].head(10)

,expected_pick_number,ev_points
0,23,22.758621
1,53,11.186441
2,23,22.758621
3,53,11.186441
4,23,22.758621
5,53,11.186441
6,23,22.758621
7,53,11.186441
8,23,22.758621
9,53,11.186441


In [8]:
CURRENT_YEAR = 2026
DISCOUNT_RATE = 0.10

future["years_out"] = (future["pick_year"] - CURRENT_YEAR).clip(lower=0)
future["discount_factor"] = 1 / ((1 + DISCOUNT_RATE) ** future["years_out"])
future["pv_ev_points"] = future["ev_points"] * future["discount_factor"]

future[["team", "pick_year", "expected_pick_number", "pv_ev_points"]].head(10)

,team,pick_year,expected_pick_number,pv_ev_points
0,Atlanta Hawks,2026,23,22.758621
1,Atlanta Hawks,2026,53,11.186441
2,Atlanta Hawks,2027,23,20.689655
3,Atlanta Hawks,2027,53,10.169492
4,Atlanta Hawks,2028,23,18.808777
5,Atlanta Hawks,2028,53,9.244992
6,Atlanta Hawks,2029,23,17.098889
7,Atlanta Hawks,2029,53,8.404538
8,Atlanta Hawks,2030,23,15.544444
9,Atlanta Hawks,2030,53,7.640490


In [9]:
team_pick_assets = (
    future.groupby("team")["pv_ev_points"]
          .sum()
          .reset_index()
          .rename(columns={"pv_ev_points": "pick_asset_value"})
          .sort_values("pick_asset_value", ascending=False)
)

team_pick_assets.head(10)

,team,pick_asset_value
29,Washington Wizards,600.450852
25,Sacramento Kings,534.821430
28,Utah Jazz,483.346607
2,Brooklyn Nets,441.809008
11,Indiana Pacers,407.522278
6,Dallas Mavericks,378.693435
18,New Orleans Pelicans,354.079491
14,Memphis Grizzlies,332.791201
4,Chicago Bulls,314.175294
16,Milwaukee Bucks,297.740853


In [10]:
combo = team_roster.merge(team_pick_assets, on="team", how="left")
combo["pick_asset_value"] = combo["pick_asset_value"].fillna(0)
combo["total_capital_proxy"] = combo["total_value"] + combo["pick_asset_value"]

combo.sort_values("total_capital_proxy", ascending=False).head(10)

,team,total_value,pick_asset_value,total_capital_proxy
0,Oklahoma City Thunder,47461.899966,151.732386,47613.632353
1,Minnesota Timberwolves,39781.800549,155.361409,39937.161958
2,Cleveland Cavaliers,39325.980282,159.181187,39485.161469
3,Los Angeles Clippers,36022.439880,163.207945,36185.647825
4,Houston Rockets,35856.830276,167.459866,36024.290142
5,Boston Celtics,34583.580315,171.957393,34755.537707
6,New York Knicks,32548.230052,176.723603,32724.953655
7,Atlanta Hawks,31754.720224,181.784653,31936.504877
8,Golden State Warriors,30555.380245,187.170319,30742.550564
9,Denver Nuggets,28738.300383,192.914654,28931.215037


In [11]:
future.to_csv(os.path.join(PROC_DIR, "future_picks_with_ev_2025_26.csv"), index=False)
team_pick_assets.to_csv(os.path.join(PROC_DIR, "team_pick_asset_value_2025_26.csv"), index=False)
combo.to_csv(os.path.join(PROC_DIR, "team_total_capital_proxy_2025_26.csv"), index=False)

print("Saved draft asset outputs.")

Saved draft asset outputs.
